# Ghidra Decompilation Test Mapping Analysis

This notebook analyzes the mapping between original C++ unit tests and Ghidra decompiled output.

In [ ]:
import pandas as pd
import json
from pathlib import Path
import matplotlib.pyplot as plt

# Set up paths
PROJECT_ROOT = Path('.').parent
GHIDRA_OUTPUT = PROJECT_ROOT / 'ghidra_output'

# Load data
df = pd.read_csv(GHIDRA_OUTPUT / 'test_mapping.csv')
df['address_int'] = df['address'].apply(lambda x: int(x, 16))
df

## Summary Statistics

In [ ]:
print("=" * 50)
print("SUMMARY STATISTICS")
print("=" * 50)
print(f"Total test cases: {len(df)}")
print(f"Total original lines: {df['original_lines'].sum():,}")
print(f"Total decompiled lines: {df['decompiled_lines'].sum():,}")
print(f"Average expansion factor: {df['expansion_factor'].mean():.1f}x")
print(f"Median expansion factor: {df['expansion_factor'].median():.1f}x")
print(f"Max expansion factor: {df['expansion_factor'].max():.1f}x ({df.loc[df['expansion_factor'].idxmax(), 'test_case']})")
print(f"Min expansion factor: {df['expansion_factor'].min():.1f}x ({df.loc[df['expansion_factor'].idxmin(), 'test_case']})")

## Expansion Factor Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of expansion factors
df_sorted = df.sort_values('expansion_factor', ascending=True)
axes[0].barh(df_sorted['test_case'], df_sorted['expansion_factor'], color='steelblue')
axes[0].set_xlabel('Expansion Factor (x)')
axes[0].set_title('Code Expansion Factor by Test Case')
axes[0].axvline(x=df['expansion_factor'].mean(), color='red', linestyle='--', label=f'Mean: {df["expansion_factor"].mean():.1f}x')
axes[0].legend()

# Scatter plot: original vs decompiled lines
axes[1].scatter(df['original_lines'], df['decompiled_lines'], s=100, alpha=0.7)
for i, row in df.iterrows():
    axes[1].annotate(row['test_case'][:15], (row['original_lines'], row['decompiled_lines']), fontsize=8)
axes[1].set_xlabel('Original Lines')
axes[1].set_ylabel('Decompiled Lines')
axes[1].set_title('Original vs Decompiled Lines')

plt.tight_layout()
plt.show()

## By Source File

In [ ]:
by_file = df.groupby('source_file').agg({
    'test_case': 'count',
    'original_lines': 'sum',
    'decompiled_lines': 'sum',
    'expansion_factor': 'mean'
}).rename(columns={'test_case': 'num_tests'})
by_file['total_expansion'] = by_file['decompiled_lines'] / by_file['original_lines']
by_file

## By Project

In [ ]:
by_project = df.groupby('project').agg({
    'test_case': 'count',
    'original_lines': 'sum',
    'decompiled_lines': 'sum',
    'expansion_factor': 'mean'
}).rename(columns={'test_case': 'num_tests'})
by_project

## Distribution of Decompiled Lines

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
df_sorted = df.sort_values('decompiled_lines', ascending=True)
colors = ['orange' if p == 'tomlplusplus' else 'steelblue' for p in df_sorted['project']]
ax.barh(df_sorted['test_case'], df_sorted['decompiled_lines'], color=colors)
ax.set_xlabel('Decompiled Lines')
ax.set_title('Decompiled Lines by Test Case (blue=inja, orange=tomlplusplus)')
plt.tight_layout()
plt.show()

## Full JSON Metadata

In [ ]:
with open(GHIDRA_OUTPUT / 'test_mapping.json') as f:
    metadata = json.load(f)

print("Projects:")
for name, info in metadata['projects'].items():
    print(f"  {name}:")
    print(f"    Binary: {info['binary']}")
    print(f"    Total decompiled lines: {info['total_decompiled_lines']:,}")
    print(f"    Test framework: {info['test_framework']}")

## Extract Decompiled Code for a Specific Test

In [ ]:
def extract_decompiled(test_case: str) -> str:
    """Extract decompiled code for a test case."""
    row = df[df['test_case'] == test_case].iloc[0]
    project = row['project']
    start = int(row['decompiled_start_line'])
    end = int(row['decompiled_end_line'])
    
    if project == 'inja':
        path = GHIDRA_OUTPUT / 'inja' / 'inja_test_decompiled.c'
    else:
        path = GHIDRA_OUTPUT / 'tomlplusplus' / 'toml_simple_test_decompiled.c'
    
    with open(path) as f:
        lines = f.readlines()
    
    return ''.join(lines[start-1:end])

# Example: extract the smallest test
print("First 50 lines of 'global-path' test:")
print(extract_decompiled('global-path')[:3000])

## Export for Further Analysis

In [ ]:
# Save enriched dataframe
df.to_csv(GHIDRA_OUTPUT / 'test_mapping_enriched.csv', index=False)
print("Saved to test_mapping_enriched.csv")